# YouTube dökümü (Colab GPU)

Uzun ders videolarını **Colab'de** yazıya döker; çıktı, yerel
`biyo-cozucu transkript-al` komutunun beklediği biçimdedir.

Neden Colab: 10 saatlik bir video Mac'te whisper.cpp ile ~100 dakika
sürüyor, 1 GB'lık WAV'ı belleğe alıyor ve HTTP çağrısındaki 1 saatlik
zaman aşımını aşıyor. T4/A100'de aynı iş dakikalar sürer.

**Akış:** Çalışma zamanı → Türü değiştir → **GPU** seç → hücreleri sırayla
çalıştır → inen `<video_id>.json` dosyalarını Mac'te şu komutla al:

```bash
uv run biyo-cozucu transkript-al ~/Downloads/<video_id>.json
uv run biyo-cozucu youtube-al "<url>" --uygula   # indirmez, dökmez; çıkarıma geçer
```

In [ ]:
#@title 1) Kurulum
# faster-whisper: CTranslate2 tabanlı, GPU'da large-v3'ü gerçek zamanın
# çok üstünde bir hızda koşturuyor. yt-dlp sık güncellendiği için sürüm
# sabitlenmiyor — YouTube extractor'ları bozulduğunda en yenisi gerekir.
!pip -q install faster-whisper==1.1.0 yt-dlp
!apt-get -qq install -y ffmpeg > /dev/null

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK — Çalışma zamanı türünü GPU yapın')

In [ ]:
#@title 2) Videolar
# Her satıra bir YouTube adresi. Playlist adresi de verilebilir.
ADRESLER = """
https://youtu.be/7SjNfe_c8As
""".split()

MODEL = "large-v3"  #@param ["large-v3", "large-v2", "medium"]
DIL = "tr"
print(f'{len(ADRESLER)} adres · model {MODEL}')

In [ ]:
#@title 3) Künye ve ses indirme
import json, subprocess, pathlib

SES_DIZINI = pathlib.Path('ses'); SES_DIZINI.mkdir(exist_ok=True)

def kunye(adres):
    ham = subprocess.run(
        ['yt-dlp', '--dump-single-json', '--flat-playlist', '--no-warnings', adres],
        capture_output=True, text=True, check=True).stdout
    veri = json.loads(ham)
    girdiler = veri.get('entries') if isinstance(veri.get('entries'), list) else [veri]
    return [g for g in girdiler if isinstance(g, dict) and g.get('id')]

videolar = []
for adres in ADRESLER:
    videolar.extend(kunye(adres))
print(f'{len(videolar)} video')
for v in videolar:
    sure = v.get('duration') or 0
    print(f"  {v['id']}  {int(sure)//60:>4} dk  {(v.get('title') or '')[:60]}")

# Ses ham indirilir: 16 kHz dönüşümünü faster-whisper kendi yapıyor,
# araya bir mp3 sokmak çift transcode olurdu.
for v in videolar:
    hedef = SES_DIZINI / f"{v['id']}.%(ext)s"
    if list(SES_DIZINI.glob(f"{v['id']}.*")):
        print('zaten inmiş:', v['id']); continue
    subprocess.run(
        ['yt-dlp', '-f', 'bestaudio/best', '--no-playlist', '--no-warnings',
         '-o', str(hedef), f"https://www.youtube.com/watch?v={v['id']}"],
        check=True)
    print('indi:', v['id'])

In [ ]:
#@title 4) Yazıya dökme
import time
from faster_whisper import WhisperModel

model = WhisperModel(MODEL, device='cuda', compute_type='float16')
CIKTI = pathlib.Path('transkript'); CIKTI.mkdir(exist_ok=True)

for v in videolar:
    hedef = CIKTI / f"{v['id']}.json"
    if hedef.exists():
        print('zaten dökülmüş:', hedef.name); continue
    sesler = list(SES_DIZINI.glob(f"{v['id']}.*"))
    if not sesler:
        print('ses yok, atlanıyor:', v['id']); continue

    basla = time.time()
    # `vad_filter`: uzun derslerde sessiz aralıklar modelin aynı cümleyi
    # tekrarlamasına yol açıyor; sessizlik atlanınca hem hızlanıyor hem
    # tekrar azalıyor.
    bolutler, bilgi = model.transcribe(
        str(sesler[0]), language=DIL, vad_filter=True, beam_size=5)

    parcalar = []
    for b in bolutler:
        metin = (b.text or '').strip()
        if metin:
            parcalar.append({'metin': metin,
                             'baslangic_sn': float(b.start),
                             'bitis_sn': float(b.end)})
        if len(parcalar) % 500 == 0 and parcalar:
            print(f"  {len(parcalar)} bölüt · {parcalar[-1]['bitis_sn']/60:.0f}. dakika", flush=True)

    # Yerel `transkript-al` komutunun beklediği şema.
    hedef.write_text(json.dumps({
        'video_id': v['id'],
        'baslik': v.get('title') or '',
        'kanal': v.get('channel') or v.get('uploader') or '',
        'sure_sn': float(bilgi.duration or v.get('duration') or 0),
        'parcalar': parcalar,
    }, ensure_ascii=False), encoding='utf-8')
    print(f"✓ {hedef.name}: {len(parcalar)} parça · {time.time()-basla:.0f} sn")

In [ ]:
#@title 5) İndir
from google.colab import files

for yol in sorted(CIKTI.glob('*.json')):
    print(yol.name, f'{yol.stat().st_size/1024:.0f} KB')
    files.download(str(yol))

print('\nMac tarafında:')
print('  uv run biyo-cozucu transkript-al ~/Downloads/*.json')
print('  uv run biyo-cozucu youtube-al "<url>" --uygula')